In [ ]:
# Import bibliotek

# 1. Podstawowe biblioteki do danych
import pandas as pd
import numpy as np

# 2. Przetwarzanie danych i podział zbioru (Scikit-learn)
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 3. Modele klasyczne (Scikit-learn)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# 4. Modele Boostingowe
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# 5. Deep Learning (TensorFlow/Keras & TabNet)
import tensorflow as tf
from keras import layers
from pytorch_tabnet.tab_model import TabNetClassifier

# 6. Metryki i ocena modelu
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, roc_curve, RocCurveDisplay,
    confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)

# 7. Wizualizacja i interpretowalność
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Opcjonalne: Ustawienie stylu wykresów
sns.set_theme(style="whitegrid")

In [ ]:
# Wczytanie danych
# Zbalansowany zbiór danych BRFSS dotyczący wskaźników zdrowia i cukrzycy (klasyfikacja binarna)
df = pd.read_csv(r"C:\Users\mkrol\PycharmProjects\GGSN_labolatory\project01\data/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")

In [ ]:
# Imputacja danych - zastępujemy brakujące wartości (NaN -> średnia arytmetyczna)
# mamy dwa główne podejścia (mean i median)

imputer = SimpleImputer(missing_values = np.nan, strategy='mean') # tworzymy imputację 
df_array = imputer.fit_transform(df) # stosujemy imputację na naszym df i zwracamy nowy array
df = pd.DataFrame(df_array, columns=df.columns) # łączymy nowy array z nazwami kolumn

In [ ]:
# Podział danych na cechy (X) i etykiety (y)

X = df.drop(columns=['Diabetes_binary']) 
y = df['Diabetes_binary']

In [ ]:
# Podział na dane treningowe i testowe

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    random_state=42, 
    test_size=0.2,
    stratify=y # lepsza stabilność modelu (dba o to, żeby proporcje w zbiorze testowych i treningowym były podobne)
    
)

In [ ]:
# Skalowanie danych, sprowadzenie danych do wspólnej skali

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Dopasowanie skalera do danych treningowych i ich transformacja
X_test_scaled = scaler.transform(X_test) # Transformacja danych testowych na podstawie parametrów ze zbioru treningowego


In [ ]:
# LightXGM - algorytm oparty na drzewach decyzyjnych.
# Dobre dla danych tabelarycznych i małych/średnich zbiorów
# Przy wybieraniu num_leaves posługuję się zasadą kciuka: num_leaves = 0,6 * (2^max_depth)
model_1 = lgb.LGBMClassifier(
    n_estimators=200, # liczba drzew
    learning_rate=0.05,
    max_depth=10, # maksymalna głębokość (2^10 maksymalnie liści)
    num_leaves=615, # liczba liści
    random_state=42,
    n_jobs=-1, # użyj wszystkich rdzeni procesora
    verbose=-1 # wyłącz komunikaty o działaniu
    # unbalanced=True # jeżeli zbiór jest niezbalansowany
)

print("Trenowanie modelu LightXGM...")
model_1.fit(X_train_scaled, y_train)
y_pred_1 = model_1.predict(X_test_scaled)
print("Trenowanie skończone...")

In [ ]:
# XGBoost - podobny model do LightXGM, starszy, wolniejszy, ale za to bardziej dokładny, odporny na przeuczenie
model_2 = xgb.XGBClassifier(
    n_estimators=200, #liczba drzew      
    learning_rate=0.05,      
    max_depth=6, # maksymalna głębokość (2^10 maksymalnie liści)          
    subsample=0.8, # nowe drzewo jest buduowane na 80% losowo     
    eval_metric='logloss', # metryka do sprawdzania jak dobrze idzie modelowi
    random_state=42,
    n_jobs=-1 # wykorzystaj wszystkie rdzenie procesora
)

print("Trenowanie modelu XGBoost...")
model_2.fit(X_train_scaled, y_train)
y_pred_2 = model_2.predict(X_test_scaled)
print("Trenowanie skończone...")

In [ ]:
# CatBoost - bardzo dobrze radzi sobie z danymi kategoralnymi
model_3 = CatBoostClassifier(
    iterations=500, # liczba drzew
    learning_rate=0.05,
    depth=6, # maksymalna glebokosc
    loss_function='Logloss', # funkcja straty
    eval_metric='AUC', # metryka do monitorowania 
    random_seed=42,
    verbose=0
)

print("Trenowanie modelu CatBoost...")
model_3.fit(X_train_scaled, y_train)
y_pred_3 = model_3.predict(X_test_scaled)
print("Trenowanie skończone...")

In [ ]:
# RandomForest - opiera się na metodzie "Bagging"
model_4 = RandomForestClassifier(
    n_estimators=200, # liczba drzew
    max_depth=None, # drzewa mogą rosnąć do momentu aż nie osiągnie jednego z dwóch stanów
    min_samples_leaf=2, # minimalna liczba obserwacji na liściu
    random_state=42,
    n_jobs=-1 # wykorzystaj wszystkie rdzenie procesora
)

print("Trenowanie modelu RandomForest...")
model_4.fit(X_train_scaled,y_train)
y_pred_4 = model_4.predict(X_test_scaled)
print("Trenowanie skończone...")

In [ ]:
# SVC - szuka optymalnej granicy (hiperpłaszczyzny) między klasami
# Bardzo skuteczny w mniejszych zbiorach, ale przy dużych danych (np. >50k wierszy) może być bardzo powolny
model_5 = SVC(
    kernel='rbf', # jądro gaussowskie, pozwala wyznaczać nieliniowe granice
    C=1.0, # parametr regularyzacji (siła kary za błędy)
    probability=True, # pozwala na uzyskanie wyników prawdopodobieństwa (potrzebne do AUC)
    random_state=42
)

print("Trenowanie modelu SVM...")
model_5.fit(X_train_scaled,y_train)
y_pred_5 = model_5.predict(X_test_scaled)
print("Trenowanie skończone...")

In [ ]:
# KNN - klasyfikuje na podstawie "głosowania" najbliższych sąsiadów w przestrzeni danych
# Prosty i intuicyjny, ale wymaga przeskalowanych danych (np. StandardScaler) i dużej pamięci RAM
model_6 = KNeighborsClassifier(n_neighbors=7)

print("Trenowanie modelu KNN...")
model_6.fit(X_train_scaled,y_train)
y_pred_6 = model_6.predict(X_test_scaled)
print("Trenowanie skończone...")

In [ ]:
# ANN - prosta sztuczna sieć neuronowa (płytka)
# Dobra do wykrywania nieliniowych zależności przy umiarkowanej liczbie danych
model_7 = tf.keras.Sequential([  
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1], )), # Dynamiczne pobranie liczby cech
    layers.Dense(32, activation='relu'),  # warstwa ukryta
    layers.Dense(1, activation='sigmoid') # wyjście: prawdopodobieństwo 0-1
])

print("Trenowanie modelu ANN...")
model_7.compile(
    optimizer='adam',                # najpopularniejszy silnik do nauki
    loss='binary_crossentropy',      # standard dla klasyfikacji Tak/Nie
    metrics=['accuracy']             # chcemy widzieć dokładność w trakcie nauki
)
# epochs=20 oznacza, że model przejdzie przez wszystkie dane 20 razy
model_7.fit(X_train_scaled, y_train, epochs=5, batch_size=32, verbose=1)
# Musimy zamienić prawdopodobieństwa na 0 lub 1 (próg 0.5)
y_pred_7_probs = model_7.predict(X_test_scaled)
y_pred_7 = (y_pred_7_probs > 0.5).astype("int32")
print("Trenowanie skończone...")

In [ ]:
# DNN - głęboka sieć neuronowa (Deep Neural Network)
# Składa się z wielu warstw, co pozwala na hierarchiczne rozumienie danych
model_8 = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

print("Trenowanie modelu ANN...")
model_8.compile(
    optimizer='adam',                # najpopularniejszy silnik do nauki
    loss='binary_crossentropy',      # standard dla klasyfikacji Tak/Nie
    metrics=['accuracy']             # chcemy widzieć dokładność w trakcie nauki
)
# epochs=20 oznacza, że model przejdzie przez wszystkie dane 20 razy
model_8.fit(X_train_scaled, y_train, epochs=5, batch_size=32, verbose=1)
# Musimy zamienić prawdopodobieństwa na 0 lub 1 (próg 0.5)
y_pred_8_probs = model_8.predict(X_test_scaled)
y_pred_8 = (y_pred_8_probs > 0.5).astype("int32")
print("Trenowanie skończone...")

In [ ]:
# TabNet - architektura Deep Learning zaprojektowana specjalnie pod tabele (Google Cloud AI)
# Wykorzystuje mechanizm atencji, aby "skupiać się" na najważniejszych cechach podobnie jak drzewa decyzyjne
model_9 = TabNetClassifier(
    n_d=8, # szerokość warstwy decyzyjnej
    n_a=8, # szerokość warstwy atencji
    n_steps=3, # liczba kroków architektury
    verbose=0 # wyciszenie komunikatów treningu
)

print("Trenowanie modelu TabNet...")

model_9.fit(
    X_train=X_train_scaled.values, 
    y_train=y_train.values.flatten(),
    eval_set=[(X_test_scaled.values, y_test.values.flatten())],
    eval_name=['test'],
    eval_metric=['accuracy'], 
    max_epochs=10,            
    batch_size=1024,           
    virtual_batch_size=128     
)

# Przy predykcji podajemy .values
y_pred_9 = model_9.predict(X_test_scaled.values)

print("Trenowanie skończone...")

In [ ]:
def compare_models(models_names, predictions, y_true):
    results = []
    
    for name, y_pred in zip(models_names, predictions):
        metrics = {
            'Model': name,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred),
            'Recall': recall_score(y_true, y_pred),
            'F1-Score': f1_score(y_true, y_pred),
            'AUC': roc_auc_score(y_true, y_pred)
        }
        results.append(metrics)
    
    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
    
    return df_results

models = ['LightXGM', 'XGBoost', 'CatBoost', 'RandomForest', 'SVM', 'KNN', 'ANN', 'DNN', 'TabNet']
y_predicted = [y_pred_1, y_pred_2, y_pred_3, y_pred_4, y_pred_5, y_pred_6, y_pred_7, y_pred_8, y_pred_9]

df_comparison = compare_models(models, y_predicted, y_test)
print(df_comparison)

In [ ]:
def plot_comparison(df):
    # Przekształcenie danych do formatu long dla Seaborn
    df_melted = df.melt(id_vars='Model', var_name='Metryka', value_name='Wynik')
    
    sns.barplot(data=df_melted, x='Model', y='Wynik', hue='Metryka')
    plt.title('Porównanie metryk dla różnych modeli')
    plt.figure(figsize=(10, 6)) 
    plt.ylim(0, 1.1) 
    plt.xticks(rotation=45)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

plot_comparison(df_comparison)

In [ ]:
def plot_all_confusion_matrices(models_names, predictions, y_true):
    n = len(models_names)
    cols = 3
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
    axes = axes.flatten()
    
    for i, (name, y_pred) in enumerate(zip(models_names, predictions)):
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues')
        axes[i].set_title(f'Macierz: {name}')
        axes[i].set_xlabel('Przewidywanie')
        axes[i].set_ylabel('Rzeczywistość')
    
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.figure(figsize=(10, 10)) 
    plt.show()

plot_all_confusion_matrices(models, y_predicted, y_test)

In [ ]:
def plot_all_shap_summary(models_dict, X_test):
    n = len(models_dict)
    cols = 2 
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 6))
    axes = axes.flatten()
    
    for i, (name, model) in enumerate(models_dict.items()):
        plt.sca(axes[i]) 
        explainer = shap.Explainer(model, X_test)
        shap_values = explainer(X_test)
        shap.summary_plot(shap_values, X_test, show=False, plot_size=None)
        axes[i].set_title(f'SHAP: {name}')
    
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.show()

tree_models_dict = {
    'LightXGM': model_1,
    'XGBoost': model_2,
    'CatBoost': model_3,
    'RandomForest': model_4
}

plot_all_shap_summary(tree_models_dict, X_test)

In [ ]:
def plot_correlation_comparison(dataframes_dict):
    n = len(dataframes_dict)
    cols = 2
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 7))
    axes = axes.flatten()
    
    for i, (name, df) in enumerate(dataframes_dict.items()):
        corr = df.corr()
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', ax=axes[i], 
                    mask=mask, cbar_kws={"shrink": .8})
        axes[i].set_title(f'Korelacje: {name}')
    
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.show()
    
data_to_compare = {
    "Original Data": df,
    "Transformed Data": X_train 
}
plot_correlation_comparison(data_to_compare)

In [ ]:
def plot_all_roc_curves(models_dict, X_test, y_test, X_test_scaled):
    n = len(models_dict)
    cols = 3
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
    axes = axes.flatten()
    
    for i, (name, model) in enumerate(models_dict.items()):
        current_X = X_test_scaled if name in ['SVM', 'KNN', 'ANN', 'DNN'] else X_test
        
        if name in ['ANN', 'DNN']:
            y_probs = model.predict(current_X).ravel()
            RocCurveDisplay.from_predictions(y_test, y_probs, ax=axes[i], name=name)
        else:
            RocCurveDisplay.from_estimator(model, current_X, y_test, ax=axes[i])
            
        axes[i].set_title(f'ROC: {name}')
        axes[i].grid(alpha=0.3)
    
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.show()

models_to_plot = {
    "LightXGM": model_1,
    "XGBoost": model_2,
    "CatBoost": model_3,
    "Random Forest": model_4,
    "SVM": model_5,
    "KNN": model_6,
    "ANN": model_7,
    "DNN": model_8,
    "TabNet": model_9
}

plot_all_roc_curves(models_to_plot, X_test, y_test, X_test_scaled)

In [ ]:
# Optymalizacja dla RandomForest

# Tworzymy parametry które będziemy sprawdzać
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500, 1000, 2000],  # liczba drzew
    'max_depth': [None, 5, 10, 15, 20, 30, 40, 50],         # głębokość (None = do oporu)
    'min_samples_split': [2, 3, 4, 5, 8, 10, 15],           # ile próbek, by podzielić węzeł
    'min_samples_leaf': [1, 2, 4,5 ],                       # ile próbek musi zostać na liściu
    'max_features': ['sqrt', 'log2'],                       # ile cech brać pod uwagę przy podziale
    'bootstrap': [True, False]                              # czy losować ze zwracaniem
}

# Tworzymy obiekt wyszukiwania
rf_random = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,     # parametry które porównujemy
    n_iter=10,                          # oznacza, że sprawdzamy 10 losowych kombinacji
    cv=3,                               # dzielimy dane na 3 części
    verbose=0,                          # nic nie wypisuje
    random_state=42, 
    n_jobs=1,                           # użyj wszystkich rdzeni
    scoring='accuracy'
)

rf_random.fit(X_train, y_train)
print("Najlepsze parametry:", rf_random.best_params_)


In [ ]:
# Definicja przestrzeni parametrów
param_dist_svc = {
    'C': [0.1, 1, 10, 100],                 # Od bardzo elastycznego do sztywnego
    'gamma': [1, 0.1, 0.01, 'scale'],       # Od bardzo "wygiętego" do gładkiego
    'kernel': ['rbf', 'poly']               # Rodzaj kształtu granicy
}

# Wybieramy tylko 5000 próbek dla optymalizacji
X_opt_sample = X_train_scaled[:5000]
y_opt_sample = y_train[:5000]

svc_random = RandomizedSearchCV(
    estimator=SVC(probability=True), 
    param_distributions=param_dist_svc,
    n_iter=10, 
    cv=3, 
    verbose=2, 
    random_state=42, 
    n_jobs=-1
)

print("Optymalizacja SVC (na próbce)...")
svc_random.fit(X_opt_sample, y_opt_sample)

print("Najlepsze parametry SVC:", svc_random.best_params_)